# Actividad Práctica: Árbol de Decisión y Random Forest para Prevenir Riesgos
**Curso:** Agentes Inteligentes - Unidad 3  
**Caso:** Predicción de Deserción Universitaria (`deserta`)  

**Documentación de la Base de Datos:**
- **Naturaleza:** Datos sintéticos ($n=200$ registros).
- **Estructura:** 5 variables numéricas predictoras (promedio ponderado, porcentaje de asistencia, cursos desaprobados, días de retraso en pensión, horas de trabajo semanal) y 1 variable objetivo binaria (Sí/No deserta).
- **Generación:** Creados computacionalmente en Python mediante distribuciones estadísticas (`np.random.normal`, `poisson`, `exponential`) que asignan probabilidades de deserción basadas en umbrales de riesgo académico y financiero.
- **Fuente de inspiración:** Modelados a partir de los indicadores educativos de retención universitaria del Minedu-SIRIES.

## Parte 1: El árbol a mano (14 correos de Phishing)

### 1. Cálculo de impureza de Gini ($Gini = 1 - \sum p_i^2$)

- **Gini total del nodo raíz (14 correos: 7 Sí, 7 No):**
  $$Gini(D) = 1 - \left((7/14)^2 + (7/14)^2\right) = 1 - (0.25 + 0.25) = 0.500$$

- **Variable Urgencia:**
  - *Alta (4 correos: 4 Sí, 0 No):* $Gini = 1 - (1^2 + 0^2) = 0.0$
  - *Media (5 correos: 3 Sí, 2 No):* $Gini = 1 - ((3/5)^2 + (2/5)^2) = 1 - (0.36 + 0.16) = 0.480$
  - *Baja (5 correos: 0 Sí, 5 No):* $Gini = 1 - (0^2 + 1^2) = 0.0$
  - **Gini ponderado Urgencia:** $(4/14)(0.0) + (5/14)(0.480) + (5/14)(0.0) = \mathbf{0.171}$

- **Variable Remitente:**
  - *Desconocido (8 correos: 5 Sí, 3 No):* $Gini = 1 - ((5/8)^2 + (3/8)^2) = 0.4688$
  - *Conocido (6 correos: 2 Sí, 4 No):* $Gini = 1 - ((2/6)^2 + (4/6)^2) = 0.4444$
  - **Gini ponderado Remitente:** $(8/14)(0.4688) + (6/14)(0.4444) = \mathbf{0.458}$

- **Variable Adjunto:**
  - *Sí (7 correos: 4 Sí, 3 No):* $Gini = 1 - ((4/7)^2 + (3/7)^2) = 0.4898$
  - *No (7 correos: 3 Sí, 4 No):* $Gini = 1 - ((3/7)^2 + (4/7)^2) = 0.4898$
  - **Gini ponderado Adjunto:** $(7/14)(0.4898) + (7/14)(0.4898) = \mathbf{0.490}$

- **Variable Enlace corto:**
  - *Sí (6 correos: 3 Sí, 3 No):* $Gini = 1 - ((3/6)^2 + (3/6)^2) = 1 - (0.25 + 0.25) = 0.500$
  - *No (8 correos: 4 Sí, 4 No):* $Gini = 1 - ((4/8)^2 + (4/8)^2) = 1 - (0.25 + 0.25) = 0.500$
  - **Gini ponderado Enlace corto:** $(6/14)(0.500) + (8/14)(0.500) = \mathbf{0.500}$

**Decisión del nodo raíz:** Se selecciona **Urgencia** por tener el menor Gini ponderado ($0.171$), siendo la variable que más reduce la impureza.

---

### 2. Reglas del árbol manual:
1. **SI** Urgencia = "Alta" **ENTONCES** Phishing = Sí
2. **SI** Urgencia = "Baja" **ENTONCES** Phishing = No
3. **SI** Urgencia = "Media" **Y** Remitente = "Desconocido" **ENTONCES** Phishing = Sí
4. **SI** Urgencia = "Media" **Y** Remitente = "Conocido" **ENTONCES** Phishing = No
5. **EN OTRO CASO** Phishing = No

*(Adjunta la foto de tu desarrollo en papel en esta misma carpeta y asegúrate de nombrarla `arbol_mano.png`)*  
![Árbol a mano](arbol_mano.png)

In [ ]:
# Parte 0: Generación de datos sintéticos (Caso Deserción) e importaciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(42)
n_filas = 200

# 5 variables numéricas predictoras
promedio_ponderado = np.random.normal(12.8, 2.8, n_filas).clip(5, 20)
porcentaje_asistencia = np.random.normal(78, 14, n_filas).clip(30, 100)
cursos_desaprobados = np.random.poisson(1.4, n_filas).clip(0, 6)
dias_retraso_pension = np.random.exponential(12, n_filas).clip(0, 90)
horas_trabajo_semana = np.random.choice([0, 10, 20, 30, 40], n_filas, p=[0.4, 0.2, 0.2, 0.1, 0.1])

# Lógica estocástica de deserción
riesgo = (
    (promedio_ponderado < 11.5) * 0.45 +
    (porcentaje_asistencia < 70) * 0.30 +
    (cursos_desaprobados >= 2) * 0.25 +
    (dias_retraso_pension > 30) * 0.20
)
deserta = np.where(np.random.rand(n_filas) < riesgo, 1, 0)

CASO, OBJETIVO = "Desercion", "deserta"
df = pd.DataFrame({
    'promedio_ponderado': promedio_ponderado.round(1),
    'porcentaje_asistencia': porcentaje_asistencia.round(0),
    'cursos_desaprobados': cursos_desaprobados,
    'dias_retraso_pension': dias_retraso_pension.round(0),
    'horas_trabajo_semana': horas_trabajo_semana,
    OBJETIVO: deserta
})

print(f"Dimensiones del dataset: {df.shape}")
df.head()

In [ ]:
# Parte 2: Tu árbol en Python
X = df.drop(columns=OBJETIVO)
y = df[OBJETIVO]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_tr, y_tr)

plt.figure(figsize=(15, 6))
plot_tree(arbol, feature_names=list(X.columns), class_names=["No", "Si"], filled=True, rounded=True)
plt.title("Árbol de Decisión (max_depth=3)")
plt.show()

print("--- Reglas del árbol ---")
print(export_text(arbol, feature_names=list(X.columns)))

# Comprobación de sobreajuste con árbol sin límite
libre = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)
print("Entrenamiento:", round(libre.score(X_tr, y_tr), 3))
print("Prueba:       ", round(libre.score(X_te, y_te), 3))

### Respuestas Parte 2:
1. **¿Qué variable eligió el árbol como raíz? ¿Tiene sentido para tu caso?**
   - El árbol eligió `promedio_ponderado` (umbral $\le 11.45$). Tiene total sentido académico y práctico, pues un rendimiento reprobatorio o al límite de la nota mínima aprobatoria es el detonante principal de riesgo y desmotivación en la carrera.

2. **Escribe dos reglas del árbol en palabras simples:**
   - *Regla 1 (Riesgo Crítico / Deserta = Sí):* SI el promedio ponderado es menor o igual a 11.45 Y los días de retraso en la pensión superan los 16.5 días Y el promedio ponderado supera los 7.85 ENTONCES el modelo predice que el estudiante desertará (Deserta = Sí con 14 de 14 casos exactos).
   - *Regla 2 (Permanencia / Deserta = No):* SI el promedio ponderado es mayor a 11.45 Y los cursos desaprobados son menores o iguales a 1.5 (es decir, máximo 1 curso jalado) Y el retraso de pensión es menor o igual a 45 días ENTONCES el modelo predice que el estudiante continuará sus estudios (Deserta = No con 60 de 63 casos).

3. **¿Por qué el árbol sin límite acierta casi el 100% en entrenamiento pero menos en prueba?**
   - Se debe al **sobreajuste (overfitting)**. Al no restringir la profundidad, el árbol genera ramas hiperespecíficas hasta memorizar casos particulares y ruido estadístico del conjunto de entrenamiento. Al enfrentarse a registros nuevos del conjunto de prueba (`X_te`), pierde capacidad de generalización y su exactitud decae notablemente.

In [ ]:
# Parte 3: Random Forest y comparación
bosque = RandomForestClassifier(n_estimators=100, random_state=42)
bosque.fit(X_tr, y_tr)

for nombre, modelo in [("ÁRBOL", arbol), ("RANDOM FOREST", bosque)]:
    pred = modelo.predict(X_te)
    print("=" * 35)
    print(nombre)
    print("=" * 35)
    print("Matriz de confusión [[VN, FP], [FN, VP]]:")
    print(confusion_matrix(y_te, pred))
    print("\nReporte de clasificación:")
    print(classification_report(y_te, pred))

plt.figure(figsize=(8, 3.5))
pd.Series(bosque.feature_importances_, index=X.columns).sort_values().plot.barh(color="#002060")
plt.title("¿Qué variables pesan más?")
plt.xlabel("Importancia")
plt.show()

### Tabla comparativa (Clase 1 = Deserta)

| Modelo | Accuracy | Recall clase 1 | F1 clase 1 | VP | FN |
| :--- | :---: | :---: | :---: | :---: | :---: |
| Árbol (profundidad 3) | 0.72 | 0.90 | 0.69 | 19 | 2 |
| Random Forest (100 árboles) | 0.78 | 0.71 | 0.70 | 15 | 6 |

#### Respuestas Parte 3:
1. **¿Qué modelo tiene mejor F1? ¿Por cuánto?**
   - El **Random Forest** obtiene un F1-score ligeramente superior (0.70 frente a 0.69 del árbol, superándolo por 0.01 o 1%)[cite: 4]. Aunque el árbol tuvo un recall mayor al ser más agresivo clasificando positivos, el Random Forest logra una precisión mucho más alta reduciendo los falsos positivos (de 15 a 7) y alcanzando una mejor exactitud global (78% frente a 72%)[cite: 4].
2. **¿Qué significa un FN en tu caso? ¿Es peor que un FP? Explica en dos líneas.**
   - Un FN significa predecir que un estudiante continuará cuando en realidad va a desertar, perdiendo la oportunidad de salvarlo[cite: 4]. Es considerablemente peor que un FP, pues intervenir con tutoría innecesaria a quien no desertará no genera un daño grave, pero omitir a un desertor real implica la pérdida definitiva del alumno[cite: 4].
3. **¿Cuáles son las dos variables que más pesan en el bosque?**
   - Según el gráfico de barras obtenido, las dos variables que más pesan son `promedio_ponderado` (con más de 0.35 de peso) y `porcentaje_asistencia` (con más de 0.20)[cite: 4, 8].

## Parte 4: Conclusión y Recomendación

Aunque el Árbol de Decisión individual obtuvo menos Falsos Negativos (2) que el Random Forest (6), esto se debió a que el árbol simple actuó de forma excesivamente agresiva al predecir "Deserta", lo que disparó sus Falsos Positivos a 15 (clasificando erróneamente a muchos estudiantes seguros como riesgo). Por esta razón, **recomendamos implementar el modelo de Random Forest**. Al ser un ensamble, logra un equilibrio superior: duplica la precisión reduciendo los Falsos Positivos a la mitad (7) y mejora la exactitud general (78% vs 72%). 

Las dos variables que la dirección académica debe vigilar estrictamente son el **promedio de notas acumulado** y el **porcentaje de inasistencias**. Como acción preventiva, se propone citar automáticamente a consejería académica a cualquier alumno que baje de 11.5 de promedio o supere el 25% de inasistencias.

## Bono (+2 Puntos): Simulación Interactiva con IA

- **Enlace público desplegado:** https://simulator-agente-sem10.netlify.app/
- **Repositorio de código:** https://github.com/Matheuu17/simulator-agente-sem10
### Cumplimiento de requerimientos del Anexo:
1. **Archivo Único Autocontenido:** Desarrollado íntegramente en HTML, CSS responsivo y JavaScript Vanilla, empleando Chart.js vía CDN público para los gráficos de importancia de variables[cite: 6].
2. **Identidad Visual y Avisos:** Paleta estructurada en Azul Marino (`#002060`) y Celeste (`#DCE6F2`), incluyendo de forma explícita el aviso reglamentario: *«Datos sintéticos con fines didácticos»*[cite: 6].
3. **Pestaña 1 (Gini Paso a Paso):** 
   - Procesa la tabla completa de 14 correos de la hoja `Phishing_mano`[cite: 2, 6].
   - Permite consultar de forma interactiva los conteos y el Gini por rama de cada variable, comprobando analíticamente que `Urgencia` obtiene el menor Gini ponderado ($0.171$) y se selecciona como nodo raíz[cite: 2, 4, 6].
   - Despliega la inducción nodo por nodo hasta generar el árbol completo con sus reglas `SI ... ENTONCES` y la regla de cierre `EN OTRO CASO`[cite: 2, 6].
4. **Pestaña 2 (Un Árbol frente a un Bosque):**
   - Generación pseudoaleatoria determinística con semilla fija ($42$) de 200 filas para los 5 temas solicitados[cite: 6].
   - Motores de inferencia en JavaScript para el Árbol (control de profundidad de 1 a 10) y Random Forest (ensamble de 1 a 200 árboles)[cite: 6].
   - Demostración dinámica del sobreajuste (overfitting) en el árbol vs. estabilidad del bosque, con matrices de confusión en tiempo real ($VP, FP, FN, VN$), exactitud en train/test, recall y F1-score[cite: 3, 4, 6].